# Input Params

In [2]:
import os
import warnings
from datetime import datetime
from pathlib import Path

# Suppress harmless odc-stac intermediate warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Dataset has no geotransform, gcps, or rpcs")

def get_crop_config(crop_name: str, year_str: str) -> dict:
    """Factory function to generate dynamic parameters per crop."""
    current_year = int(year_str)
    prev_year = current_year - 1
    
    configs = {
        "cane": {
            "Target_Crop_Class": [1],
            "min_pixel_size": 20,
            "min_area_acres": 0.5,
            "target_class_in": 1,
            "target_class_out": 1,
            "background_out": 4,
            "keep_label": 1,
            "relabel_as": 1,
            "NDVI_START": f"{prev_year}-11-24",
            "NDVI_END": f"{current_year}-09-09",
            "inference_start_date": f"{prev_year}-12-09",
            "inference_end_date": f"{current_year}-09-08",
            "base_start": f"{current_year}-10-07",
            "base_end": f"{current_year}-10-15",
            "MODEL_FILE_PATH": f"/mnt/c/Work_Work_Work/Python/Scripts/cropscan/model_files/best_rf_classifier_v4.joblib",
            "static_image_model": f"/mnt/c/Work_Work_Work/Python/Scripts/cropscan/model_files/fao_cane_xgb_model_v4.json"
        },
    }
    
    if crop_name not in configs:
        raise ValueError(f"Pipeline Error: Configuration for '{crop_name}' is not defined.")
        
    return configs[crop_name]


In [4]:
# =====================================================================
# GLOBAL EXECUTION TOGGLES
# =====================================================================
crop = 'cane'  # Change to 'spring_maize' or 'wheat' to route entirely
year = '2026' 
district_name = "almoiz_unit_1_test_feature_1"
run_static_model = True
static_image_date = ["2026-08-30"]

# Dynamic System Paths
input_shp_path = f"/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/{district_name}/{district_name}.shp"
final_output_name = f'{district_name}_{crop}_{year}_new'

# Extract specific crop configuration
cfg = get_crop_config(crop, year)

Target_Crop_Class = cfg["Target_Crop_Class"]
min_pixel_size = cfg["min_pixel_size"]
min_area_acres = cfg["min_area_acres"]
target_class_in = cfg["target_class_in"]
target_class_out = cfg["target_class_out"]
background_out = cfg["background_out"]
keep_label = cfg["keep_label"]
relabel_as = cfg["relabel_as"]

NDVI_START = cfg["NDVI_START"]
NDVI_END = cfg["NDVI_END"]
inference_start_date = cfg["inference_start_date"]
inference_end_date = cfg["inference_end_date"]

# Convert string dates to datetime objects for STAC
base_start = datetime.strptime(cfg["base_start"], '%Y-%m-%d')
base_end = datetime.strptime(cfg["base_end"], '%Y-%m-%d')
STATIC_START = base_start.strftime('%Y-%m-%d')
STATIC_END = base_end.strftime('%Y-%m-%d')

MODEL_FILE_PATH = cfg["MODEL_FILE_PATH"]
static_image_model = cfg["static_image_model"]

# Acquisition Constants
EXPORT_SCALE = 10 
STEP_DAYS = 8
TILE_DEG = 0.1 
STATIC_BANDS = ['B2', 'B3', 'B4', 'B5', 'B8', 'NDVI']

save_shp_zip = True

print(f"Active Crop Target: {crop.upper()}")
print(f"Export Scale: {EXPORT_SCALE}m | Tile Grid: {TILE_DEG}°")
print(f"NDVI Time-Series: {NDVI_START} to {NDVI_END}")
print(f"Static Img Anchor: {STATIC_START} to {STATIC_END}")

Active Crop Target: CANE
Export Scale: 10m | Tile Grid: 0.1°
NDVI Time-Series: 2025-11-24 to 2026-09-09
Static Img Anchor: 2026-10-07 to 2026-10-15


# NDVI based Model Prediction

In [5]:
%%writefile geo_inference_workers.py
import os
import gc
from pathlib import Path
from typing import List, Optional, Tuple, Any, Dict

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

def parse_stac_bands(descriptions: Tuple[str, ...]) -> Tuple[List[int], List[int], List[str]]:
    """Parses red and nir bands generated dynamically by sentinel.py STAC fetcher."""
    red_idx, nir_idx, dates = [], [], []
    for i, d in enumerate(descriptions):
        if not d: continue
        if d.startswith('red_'):
            red_idx.append(i)
            parts = d.split('_')
            dates.append(f"{parts[1]}-{parts[2]}-{parts[3]}")
        elif d.startswith('nir_'):
            nir_idx.append(i)
    assert len(red_idx) == len(nir_idx), "Mismatch between Red and NIR band counts."
    return red_idx, nir_idx, dates

def get_penalty_matrix(n: int, lmbd: float, d: int) -> np.ndarray:
    E = np.eye(n)
    D = np.diff(E, n=d, axis=0)
    return lmbd * (D.T @ D)

def whittaker_pixel(y: np.ndarray, penalty_mat: np.ndarray) -> np.ndarray:
    mask = np.isfinite(y)
    if not mask.any(): return y
    W = np.diag(mask.astype(float))
    A = W + penalty_mat
    rhs = np.zeros_like(y)
    rhs[mask] = y[mask]
    try: 
        return np.linalg.solve(A, rhs)
    except np.linalg.LinAlgError: 
        return np.linalg.lstsq(A, rhs, rcond=None)[0]

def process_smoothing_chunk(
    data_chunk: np.ndarray, penalty_mat: np.ndarray,
    clip_bounds: Optional[Tuple[float, float]], nodata_val: Optional[float]
) -> np.ndarray:
    B, H, W_w = data_chunk.shape
    pixels = data_chunk.transpose(1, 2, 0).reshape(-1, B).astype(np.float32)
    
    missing_mask = np.isnan(pixels) if nodata_val is None or np.isnan(nodata_val) else (pixels == nodata_val)
    missing_counts = missing_mask.sum(axis=1)
    
    full_valid_idx = np.where(missing_counts == 0)[0]
    partial_valid_idx = np.where((missing_counts > 0) & (missing_counts < B))[0]
    
    if len(full_valid_idx) > 0:
        A_full = np.eye(B) + penalty_mat
        try: 
            pixels[full_valid_idx] = np.linalg.solve(A_full, pixels[full_valid_idx].T).T
        except np.linalg.LinAlgError: 
            pixels[full_valid_idx] = np.linalg.lstsq(A_full, pixels[full_valid_idx].T, rcond=None)[0].T

    if len(partial_valid_idx) > 0:
        pixels[partial_valid_idx] = np.apply_along_axis(whittaker_pixel, 1, pixels[partial_valid_idx], penalty_mat=penalty_mat)
        
    if clip_bounds:
        processed_idx = np.concatenate([full_valid_idx, partial_valid_idx])
        if len(processed_idx) > 0: 
            pixels[processed_idx] = np.clip(pixels[processed_idx], *clip_bounds)
            
    return pixels.reshape(H, W_w, B).transpose(2, 0, 1)

def worker_process_local_tile(
    raw_local_path: Path, out_dir: Path, rf_model: Any, 
    inference_start_date: str, inference_end_date: str, lmbd: float, d: int, 
    clip_bounds: Tuple[float, float], output_nodata: int = 255,
    export_raw: bool = False, export_smoothed: bool = False
) -> Dict[str, Any]:
    """Processes a single STAC tile stored locally: Reads, Smooths, Predicts, and saves."""
    
    # Sandboxing GDAL environment per-process prevents C-level cache/thread corruption
    with rasterio.Env(GDAL_NUM_THREADS='1', OMP_NUM_THREADS='1', NUMEXPR_NUM_THREADS='1'):
        grid_filename = raw_local_path.name
        pred_local_path = out_dir / grid_filename.replace('.tif', '_predicted.tif')
        pred_tmp_path = pred_local_path.with_suffix('.tmp.tif')
        
        smoothed_local_path = out_dir / grid_filename.replace('.tif', '_smoothed.tif') if export_smoothed else None
        smoothed_tmp_path = smoothed_local_path.with_suffix('.tmp.tif') if export_smoothed else None

        # Robust Resume Logic: Physically reads a 1x1 block to verify the LZW stream isn't truncated
        if pred_local_path.exists():
            try:
                with rasterio.open(pred_local_path) as chk:
                    _ = chk.read(1, window=Window(0, 0, 1, 1)) 
                return {"pred": pred_local_path, "raw": raw_local_path if export_raw else None, "smoothed": smoothed_local_path}
            except Exception:
                pred_local_path.unlink(missing_ok=True)

        try:
            with rasterio.open(raw_local_path) as src:
                red_idx, nir_idx, raster_dates = parse_stac_bands(src.descriptions)
                dt_index = pd.to_datetime(raster_dates, errors='coerce')
                
                target_mask = (dt_index >= pd.to_datetime(inference_start_date)) & (dt_index <= pd.to_datetime(inference_end_date))
                target_band_indices = np.where(target_mask)[0]
                
                n_timesteps = len(raster_dates)
                penalty_mat = get_penalty_matrix(n_timesteps, lmbd, d)

                # Explicitly define clean output profiles to prevent inheriting conflicting STAC metadata
                out_profile = {
                    "driver": "GTiff",
                    "height": src.height,
                    "width": src.width,
                    "transform": src.transform,
                    "crs": src.crs,
                    "dtype": rasterio.uint8,
                    "count": 1,
                    "nodata": output_nodata,
                    "compress": "lzw",
                    "tiled": True,
                    "blockxsize": 256,
                    "blockysize": 256,
                    "predictor": 2,
                    "bigtiff": "YES"
                }

                smoothed_dst = None
                if export_smoothed:
                    smooth_meta = out_profile.copy()
                    smooth_meta.update({
                        "dtype": "float32", 
                        "count": n_timesteps
                    })
                    smoothed_dst = rasterio.open(smoothed_tmp_path, 'w', **smooth_meta)
                    for i, date_str in enumerate(raster_dates, 1):
                        smoothed_dst.set_band_description(i, f"NDVI_{date_str.replace('-', '_')}")

                try:
                    with rasterio.open(pred_tmp_path, 'w', **out_profile) as dst:
                        for _, window in src.block_windows(1):
                            raw_chunk = src.read(window=window)
                            
                            red = raw_chunk[red_idx, :, :].astype(np.float32)
                            nir = raw_chunk[nir_idx, :, :].astype(np.float32)
                            
                            denom = nir + red
                            valid_px_mask = denom > 0
                            ndvi_chunk = np.full_like(red, np.nan)
                            np.divide(nir - red, denom, out=ndvi_chunk, where=valid_px_mask)

                            smoothed_chunk = process_smoothing_chunk(ndvi_chunk, penalty_mat, clip_bounds, nodata_val=np.nan)
                            
                            if smoothed_dst:
                                smoothed_dst.write(smoothed_chunk, window=window)
                            
                            sliced_data = smoothed_chunk[target_band_indices, :, :]
                            B, H, W = sliced_data.shape
                            pixels = sliced_data.transpose(1, 2, 0).reshape(-1, B)
                            
                            valid_mask = ~np.all(np.isnan(pixels) | (pixels == 0), axis=1)
                            pred_block = np.full(pixels.shape[0], output_nodata, dtype=np.uint8)
                            
                            if valid_mask.any():
                                valid_pixels = np.nan_to_num(pixels[valid_mask], nan=0.0) 
                                pred_block[valid_mask] = rf_model.predict(valid_pixels).astype(np.uint8)
                                
                            dst.write(pred_block.reshape(H, W), 1, window=window)
                            
                    pred_tmp_path.rename(pred_local_path)
                    
                finally:
                    if smoothed_dst:
                        smoothed_dst.close()
                        if smoothed_tmp_path and smoothed_tmp_path.exists():
                            smoothed_tmp_path.rename(smoothed_local_path)
                        
        finally:
            if pred_tmp_path.exists(): pred_tmp_path.unlink()
            if smoothed_tmp_path and smoothed_tmp_path.exists(): smoothed_tmp_path.unlink()
            gc.collect()
                
        return {
            "pred": pred_local_path, 
            "raw": raw_local_path if export_raw else None, 
            "smoothed": smoothed_local_path if export_smoothed else None
        }

Overwriting geo_inference_workers.py


In [6]:
import os
import sys
import shutil
import logging
import multiprocessing
import ctypes
from pathlib import Path
from typing import Tuple, List, Optional
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import rasterio
from rasterio.merge import merge
from osgeo import gdal
from tqdm import tqdm  # Reverted to standard tqdm for strict stdout control
import joblib
import gc

from geo_inference_workers import worker_process_local_tile
from sentinel import fetch_sentinel_imagery
# ==============================================================================
# FORCE LOGGING VISIBILITY IN JUPYTER
# ==============================================================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", force=True)
logger = logging.getLogger("PipelineOrchestrator")

# Prevent multiprocessing deadlocks
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['GDAL_NUM_THREADS'] = '1'

# Enable GDAL exceptions but suppress harmless PROJ/C-level warnings
gdal.UseExceptions()
gdal.PushErrorHandler('CPLQuietErrorHandler')

def release_os_memory() -> None:
    """Forces Python and the OS (glibc) to release unreferenced memory."""
    gc.collect()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except OSError:
        pass

def mosaic_continuous_rasters(input_paths: List[Path], output_path: Path) -> None:
    logger.info(f"\nMosaicing {len(input_paths)} continuous rasters using rasterio merge...")
    src_files = [rasterio.open(fp) for fp in input_paths]
    try:
        template_src = src_files[0]
        nodata_val = template_src.nodata
        if nodata_val is None:
            is_float = np.issubdtype(template_src.dtypes[0], np.floating)
            nodata_val = np.nan if is_float else 0
            
        mosaic, out_trans = merge(src_files, res=template_src.res, nodata=nodata_val, method='first')
        
        out_meta = template_src.meta.copy()
        out_meta.update({
            "driver": "GTiff", "height": mosaic.shape[1], "width": mosaic.shape[2],
            "transform": out_trans, "nodata": nodata_val, "compress": "lzw",
            "tiled": True, "blockxsize": 256, "blockysize": 256,
            "bigtiff": "YES"
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic)
            dest.update_tags(**template_src.tags())
            for i, desc in enumerate(template_src.descriptions, start=1):
                if desc: dest.set_band_description(i, desc)
        logger.info(f"Successfully generated mosaic: {output_path}")
    finally:
        for src in src_files: src.close()
        release_os_memory()

def mosaic_categorical_rasters(input_paths: List[Path], output_path: Path, band_name: str = "Classification") -> None:
    logger.info(f"\nMosaicing {len(input_paths)} categorical chunks ({band_name}) using rasterio merge...")
    src_files = [rasterio.open(fp) for fp in input_paths]
    try:
        template_src = src_files[0]
        nodata_val = template_src.nodata or 255
            
        mosaic, out_trans = merge(src_files, res=template_src.res, nodata=nodata_val, method='first')
        
        out_meta = template_src.meta.copy()
        out_meta.update({
            "driver": "GTiff", "height": mosaic.shape[1], "width": mosaic.shape[2],
            "transform": out_trans, "nodata": nodata_val, "compress": "lzw",
            "tiled": True, "blockxsize": 256, "blockysize": 256,
            "bigtiff": "YES"
        })

        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(mosaic)
            dest.update_tags(**template_src.tags())
            dest.set_band_description(1, band_name)
                    
        logger.info(f"Successfully generated final categorical map: {output_path}")
    finally:
        for src in src_files: src.close()
        release_os_memory()

def generate_global_index_mask(reference_path: Path, out_path: Path, nodata_val: int = -1) -> None:
    """Already memory-safe (O(1) block-windowed reading)."""
    with rasterio.open(reference_path) as src:
        meta = src.profile.copy()
        W = src.width
        ref_nodata = src.nodata if src.nodata is not None else 255
        meta.update(dtype=rasterio.int32, count=1, nodata=nodata_val, compress="lzw", tiled=True, blockxsize=256, blockysize=256, bigtiff="YES")

        with rasterio.open(out_path, "w", **meta) as dst:
            for _, window in src.block_windows(1):
                data = src.read(1, window=window)
                valid_mask = (data != ref_nodata)
                
                rows, cols = np.indices((window.height, window.width))
                global_rows = rows + window.row_off
                global_cols = cols + window.col_off
                global_1d_indices = (global_rows * W + global_cols).astype(np.int32)
                
                index_block = np.full(data.shape, nodata_val, dtype=np.int32)
                index_block[valid_mask] = global_1d_indices[valid_mask]
                dst.write(index_block, 1, window=window)

# ==============================================================================
# PIPELINE ORCHESTRATOR 
# ==============================================================================
def execute_stac_inference_pipeline(
    input_shp_path: str, model_path: str, final_out_dir: str, output_basename: str, 
    inference_start_date: str, inference_end_date: str, lmbd: float = 0.5, d: int = 2,
    clip_bounds: Tuple[float, float] = (-1.0, 1.0), n_jobs: int = -1,
    export_raw_mosaic: bool = False, export_smoothed_mosaic: bool = False,
    export_index_mask: bool = False, delete_raw_tiles: bool = True
):
    out_dir = Path(final_out_dir)
    raw_tiles_dir = out_dir / "raw_stac_tiles"
    chunks_dir = out_dir / "processing_chunks"
    
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_tiles_dir.mkdir(exist_ok=True)
    chunks_dir.mkdir(exist_ok=True)
    
    final_class_path = out_dir / f"{output_basename}_rf_classification_map.tif"
    index_map_path = out_dir / f"{output_basename}_index_mask.tif"
    
    if not final_class_path.exists():
        logger.info("--- PHASE 1: STAC IMAGERY ACQUISITION (I/O BOUND) ---")
        logger.info("NOTE: Downloading time-series chunks. This may take several minutes. Please wait for Phase 2...")
        
        fetch_sentinel_imagery(
            aoi=input_shp_path,
            start=NDVI_START,
            end=NDVI_END,
            bands=["red", "nir"],
            out_dir=str(raw_tiles_dir),
            step=STEP_DAYS,
            res_m=EXPORT_SCALE,
            tile_deg=TILE_DEG,
            cloud_lt=97,
            workers=4, 
            build_vrt_mosaic=False, 
            clip_to_aoi=False
        )
        
        raw_tifs = list(raw_tiles_dir.glob("sentinel_*m_tile_*.tif"))
        if not raw_tifs:
            raise FileNotFoundError("STAC acquisition failed to output tiles.")
            
        logger.info(f"--- PHASE 2: DISTRIBUTED INFERENCE ({len(raw_tifs)} Grids) ---")
        total_cores = multiprocessing.cpu_count()
        usable_cores = max(1, int(total_cores * 0.75)) if n_jobs == -1 else n_jobs
        
        rf_model = joblib.load(model_path)
        rf_model.n_jobs = 1  
        
        processed_results = []
        
        with ProcessPoolExecutor(max_workers=usable_cores, max_tasks_per_child=1) as executor:
            futures = {
                executor.submit(
                    worker_process_local_tile, tif_path, chunks_dir, rf_model,
                    inference_start_date, inference_end_date, lmbd, d, clip_bounds, 255,
                    export_raw_mosaic, export_smoothed_mosaic
                ): tif_path
                for tif_path in raw_tifs
            }
            
            # Using strict sys.stdout and dynamic_ncols to prevent multi-line rendering
            for future in tqdm(as_completed(futures), total=len(futures), desc="Processing Grids", file=sys.stdout, dynamic_ncols=True, leave=True):
                tif_path = futures[future]
                try:
                    result = future.result(timeout=600)
                    processed_results.append(result)
                except Exception as e:
                    logger.error(f"[ERROR ENCOUNTERED] Error processing {tif_path.name}: {e}")
        
        del rf_model
        release_os_memory()

        logger.info("--- PHASE 3: MOSAICING (Code 1 rasterio.merge logic) ---")
        try:
            pred_paths = [res["pred"] for res in processed_results if res["pred"] is not None]
            if pred_paths: 
                mosaic_categorical_rasters(pred_paths, final_class_path, "RF_Classification")
            
            if export_raw_mosaic:
                raw_paths = [res["raw"] for res in processed_results if res["raw"] is not None]
                if raw_paths: 
                    mosaic_continuous_rasters(raw_paths, out_dir / f"{output_basename}_raw_mosaic.tif")
                
            if export_smoothed_mosaic:
                smoothed_paths = [res["smoothed"] for res in processed_results if res["smoothed"] is not None]
                if smoothed_paths: 
                    mosaic_continuous_rasters(smoothed_paths, out_dir / f"{output_basename}_smoothed_mosaic.tif")
            
            if delete_raw_tiles:
                if final_class_path.exists():
                    logger.info("--- CLEANING UP RAW STAC TILES ---")
                    shutil.rmtree(raw_tiles_dir, ignore_errors=True)
                else:
                    logger.warning("--- SKIPPING RAW TILE CLEANUP: Final map was not produced. Tiles retained for debugging. ---")

        finally:
            logger.info("--- CLEANING UP TEMPORARY CHUNKS ---")
            shutil.rmtree(chunks_dir, ignore_errors=True)
            release_os_memory()
            
    else:
        logger.info(f"[CHECKPOINT FOUND] Classification map already exists at: {final_class_path}")

    if export_index_mask:
        if final_class_path.exists() and not index_map_path.exists():
            generate_global_index_mask(final_class_path, index_map_path)
            release_os_memory()

    if not final_class_path.exists():
        logger.error(f"PIPELINE FAILED: The final output map was NOT produced.")
        return None
            
    return final_class_path


In [7]:
shp_path_obj = Path(input_shp_path)
new_out_dir_path: Path = shp_path_obj.parent
output_basename: str = shp_path_obj.stem

NEW_OUT_DIR = str(new_out_dir_path / f"{crop}_{year}")
logger.info(f"Output Directory: {NEW_OUT_DIR}")
logger.info(f"Basename: {output_basename}")

if __name__ == "__main__":
    try:
        final_output_path = execute_stac_inference_pipeline(
            input_shp_path=input_shp_path,
            model_path=MODEL_FILE_PATH,
            final_out_dir=NEW_OUT_DIR,
            output_basename=output_basename,
            inference_start_date=inference_start_date,
            inference_end_date=inference_end_date,
            lmbd=0.5, d=2, clip_bounds=(-1.0, 1.0),
            n_jobs=-1,  
            export_raw_mosaic=True,
            export_smoothed_mosaic=True, 
            export_index_mask=True,
            delete_raw_tiles=False
        )
        logger.info(f"Final RF Predictions available at: {final_output_path}")
    finally:
        release_os_memory()

2026-09-09 17:49:59,848 - PipelineOrchestrator - INFO - Output Directory: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026
2026-09-09 17:49:59,855 - PipelineOrchestrator - INFO - Basename: almoiz_unit_1_test_feature_1
2026-09-09 17:49:59,900 - PipelineOrchestrator - INFO - [CHECKPOINT FOUND] Classification map already exists at: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/almoiz_unit_1_test_feature_1_rf_classification_map.tif
2026-09-09 17:49:59,910 - PipelineOrchestrator - INFO - Final RF Predictions available at: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/almoiz_unit_1_test_feature_1_rf_classification_map.tif


# Seive Smoothing

In [8]:
import numpy as np
import rasterio
from rasterio.features import sieve
from scipy.ndimage import label, binary_dilation, generate_binary_structure
from pathlib import Path
from typing import List

def apply_strict_directional_sieve(
    input_raster_path: str,
    target_classes: List[int],
    min_pixel_size: int = 15,
    connectivity: int = 4,
    nodata_val: int = 255
) -> str:
    """
    Applies a strict asymmetric Sieve filter using morphological connected components.
    A Non-Target clump is ONLY allowed to merge into the Target class group if it is 
    completely surrounded by Target pixels. Touching NoData or any other class aborts the merge.
    """
    in_path = Path(input_raster_path)
    out_name = f"{in_path.stem}_strict_sieve_multiclass_p{min_pixel_size}{in_path.suffix}"
    out_path = in_path.parent / out_name

    # --- NEW: Skip processing if output already exists ---
    if out_path.exists():
        print(f"[Skipped] Strict Sieved raster already exists at:\n{out_path}")
        return str(out_path)

    print(f"Loading categorical map for Strict Sieving: {in_path.name}")
    with rasterio.open(in_path) as src:
        meta = src.profile
        data = src.read(1)
        
        if data.dtype != np.uint8:
            data = data.astype(np.uint8)
            meta.update(dtype=rasterio.uint8)

    valid_mask = (data != nodata_val).astype(np.uint8)

    print(f"Phase 1: Base Sieve Filter (Removing blobs < {min_pixel_size} pixels)...")
    sieved_data = sieve(
        data, 
        size=min_pixel_size, 
        connectivity=connectivity, 
        mask=valid_mask
    )

    print("Phase 2: Enforcing Strict Topological Encapsulation...")
    
    # Define topological boundaries using the entire group of target classes
    is_target_orig = np.isin(data, target_classes)
    is_target_sieved = np.isin(sieved_data, target_classes)
    
    # 1. Identify all pixels that changed from Non-Target -> ANY Target Class
    changed_mask = (~is_target_orig) & is_target_sieved
    
    # 2. Define "Bad Neighbors": Any pixel in original data that is NOT in the target group.
    # We exclude the `changed_mask` pixels themselves so clumps don't flag themselves.
    bad_neighbors = (~is_target_orig) & ~changed_mask
    
    # 3. Create a structuring element that matches your sieve connectivity
    struct = generate_binary_structure(2, 1) if connectivity == 4 else generate_binary_structure(2, 2)
    
    # 4. Dilate the bad neighbors by 1 pixel to create a "collision zone"
    bad_borders = binary_dilation(bad_neighbors, structure=struct)
    
    # 5. Find pixels inside our changed clumps that intersect the collision zone
    touched_by_bad = changed_mask & bad_borders
    
    # 6. Assign a unique ID to every distinct clump in the changed_mask
    labeled_clumps, num_features = label(changed_mask, structure=struct)
    
    if num_features > 0:
        bad_labels = np.unique(labeled_clumps[touched_by_bad])
        bad_labels = bad_labels[bad_labels != 0] 
        
        if len(bad_labels) > 0:
            print(f"  -> Reverting {len(bad_labels)} illegal edge-merges...")
            revert_mask = np.isin(labeled_clumps, bad_labels)
            sieved_data[revert_mask] = data[revert_mask]

    # Strictly enforce NoData limits
    sieved_data[valid_mask == 0] = nodata_val

    print("Writing topologically-enforced output to disk...")
    meta.update(compress='lzw')
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(sieved_data, 1)

    print(f"SUCCESS: Strict Sieved raster saved to:\n{out_path}")
    return str(out_path)





In [9]:
if __name__ == "__main__":
    
    sieved_map_path = apply_strict_directional_sieve(
        input_raster_path=final_output_path,
        target_classes=Target_Crop_Class,
        min_pixel_size=min_pixel_size, 
        connectivity=4,    
        nodata_val=255
    )
    
    print(f"\nReturned Path for pipeline continuation:\n{sieved_map_path}")

[Skipped] Strict Sieved raster already exists at:
/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/almoiz_unit_1_test_feature_1_rf_classification_map_strict_sieve_multiclass_p20.tif

Returned Path for pipeline continuation:
/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/almoiz_unit_1_test_feature_1_rf_classification_map_strict_sieve_multiclass_p20.tif


# Static IMG Classifier

In [10]:
import os
import sys
import gc
import ctypes
import logging
import warnings
import shutil
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from osgeo import gdal
import xgboost as xgb
from tqdm import tqdm
from typing import Any, Optional, Sequence
from pathlib import Path
from datetime import datetime
from sentinel import fetch_sentinel_static_imagery
from static_training import inference as st_inference

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

def release_os_memory() -> None:
    """Forces Python, GDAL, and the OS (glibc) to release unreferenced memory."""
    gdal.SetCacheMax(0) 
    gc.collect()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except OSError:
        pass

def configure_model_hardware(model: Any) -> Any:
    logger.info("Detecting hardware acceleration capabilities...")
    dummy_data = np.zeros((1, 6), dtype=np.float32)

    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        try:
            model.set_params(device="cuda")
            model.predict(dummy_data)
            
            fell_back = any("Device is changed from GPU to CPU" in str(warn.message) or 
                            "No visible GPU is found" in str(warn.message) for warn in w)
            
            if not fell_back:
                logger.info("Hardware bound: GPU (CUDA via device='cuda')")
                return model
        except Exception:
            pass

    model.set_params(device="cpu", n_jobs=-1)
    logger.info("Hardware bound: CPU (Utilizing all available cores for fast processing)")
    return model

def _assert_grid_parity(path_a: str, path_b: str) -> None:
    """Hard gate: two rasters must share CRS, transform, and dimensions."""
    with rasterio.open(path_a) as a, rasterio.open(path_b) as b:
        assert a.crs == b.crs, f"CRS mismatch: {a.crs} vs {b.crs}"
        assert (a.width, a.height) == (b.width, b.height), (
            f"Dimension mismatch: {a.width}x{a.height} vs {b.width}x{b.height}"
        )
        assert np.allclose(tuple(a.transform), tuple(b.transform), atol=1e-9), (
            f"Transform mismatch:\n{a.transform}\nvs\n{b.transform}"
        )

def create_aligned_mask(
    target_raster_path: str,
    raw_mask_path: str,
    shp_path: str,
    output_mask_path: str,
    keep_values: Sequence[int] = (1,),
    chunk_size: int = 2048,
    max_coverage_fraction: float = 0.90
) -> None:
    """
    Builds a 1:1 aligned binary mask clipped to the AOI and specific class labels.
    """
    logger.info(f"Generating aligned mask | keep_values={tuple(keep_values)} | clip={shp_path}")

    gdf = gpd.read_file(shp_path)
    assert not gdf.empty, "Pipeline Error: Input shapefile is empty."
    assert gdf.is_valid.all(), "Pipeline Error: Invalid geometries in clip shapefile."

    with rasterio.open(target_raster_path) as trg:
        trg_profile = trg.profile.copy()
        trg_crs = trg.crs
        trg_transform = trg.transform
        trg_width, trg_height = trg.width, trg.height

    if gdf.crs != trg_crs:
        gdf = gdf.to_crs(trg_crs)
    geometries = gdf.geometry.values

    # THE FIX: Explicitly enforce GTiff driver and add tiling for optimized downstream reading
    trg_profile.update(
        driver="GTiff",
        count=1, 
        dtype=rasterio.uint8, 
        nodata=0, 
        compress="lzw", 
        tiled=True,
        blockxsize=256,
        blockysize=256,
        bigtiff="YES"
    )
    
    keep_arr = np.asarray(keep_values)
    kept_px = 0
    total_px = trg_width * trg_height

    with rasterio.open(raw_mask_path) as src:
        vrt_options = {
            "resampling": Resampling.nearest,
            "crs": trg_crs,
            "transform": trg_transform,
            "height": trg_height,
            "width": trg_width,
            "nodata": src.nodata if src.nodata is not None else 0,
        }

        with WarpedVRT(src, **vrt_options) as vrt, rasterio.open(output_mask_path, "w", **trg_profile) as dst:
            y_offsets = range(0, trg_height, chunk_size)
            x_offsets = range(0, trg_width, chunk_size)
            n_blocks = len(list(y_offsets)) * len(list(x_offsets))

            with tqdm(total=n_blocks, desc="Clipping Mask", unit="block", file=sys.stdout, dynamic_ncols=True) as pbar:
                for y in range(0, trg_height, chunk_size):
                    for x in range(0, trg_width, chunk_size):
                        win_h = min(chunk_size, trg_height - y)
                        win_w = min(chunk_size, trg_width - x)
                        win = Window(x, y, win_w, win_h)

                        mask_chunk = vrt.read(1, window=win)
                        win_transform = rasterio.windows.transform(win, trg_transform)

                        geom_mask = geometry_mask(
                            geometries,
                            out_shape=(win_h, win_w),
                            transform=win_transform,
                            invert=True,
                            all_touched=False,
                        )

                        class_mask = np.isin(mask_chunk, keep_arr)
                        final_mask = (class_mask & geom_mask).astype(rasterio.uint8)
                        
                        kept_px += int(final_mask.sum())
                        dst.write(final_mask, 1, window=win)
                        pbar.update(1)

    coverage = kept_px / total_px
    logger.info(f"Mask coverage: {kept_px:,} px ({coverage:.2%} of target grid)")
    
    # Allow 0 coverage for areas where the target crop genuinely isn't present
    if coverage == 0:
        logger.warning(f"Degenerate mask: zero pixels kept. Target classes {keep_values} were not found in this AOI.")
        
    assert coverage < max_coverage_fraction, (
        f"Degenerate mask: {coverage:.2%} coverage exceeds {max_coverage_fraction:.0%}. "
        "Binarization is likely admitting background classes."
    )

    _assert_grid_parity(target_raster_path, output_mask_path)
    logger.info("Grid parity verified: mask is 1:1 with input image.")

def classify_large_image(
    input_path: str, 
    output_path: str, 
    mask_raster_path: Optional[str],
    input_shp_path: Optional[str],
    model: Any, 
    chunk_size: int = 2048,
    mask_keep_values: Sequence[int] = (1,),
    use_mask: bool = True,
    target_class_in: int = 1,
    target_class_out: int = 1,
    background_out: int = 0
) -> None:
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    aligned_mask_path = None

    if use_mask:
        assert mask_raster_path and input_shp_path, (
            "use_mask=True requires both mask_raster_path and input_shp_path."
        )
        aligned_mask_path = str(Path(output_path).parent / f"{Path(output_path).stem}_temp_mask.tif")
        create_aligned_mask(
            target_raster_path=input_path,
            raw_mask_path=mask_raster_path,
            shp_path=input_shp_path,
            output_mask_path=aligned_mask_path,
            keep_values=mask_keep_values,
            chunk_size=chunk_size,
        )

    with rasterio.open(input_path) as src:
        mask_src = rasterio.open(aligned_mask_path) if use_mask else None
        try:
            profile = src.profile.copy()
            width, height = src.width, src.height
            bands = src.count
            nodata_val = src.nodata if src.nodata is not None else 0

            profile.update(
                driver="GTiff", dtype=rasterio.uint8, count=1, nodata=background_out,
                compress="lzw", tiled=True, blockxsize=256, blockysize=256, bigtiff="YES"
            )

            with rasterio.open(output_path, "w", **profile) as dst:
                y_offsets = list(range(0, height, chunk_size))
                x_offsets = list(range(0, width, chunk_size))
                total_blocks = len(y_offsets) * len(x_offsets)

                with tqdm(total=total_blocks, desc="Classifying Raster Blocks", unit="block", file=sys.stdout, dynamic_ncols=True) as pbar:
                    for y in y_offsets:
                        for x in x_offsets:
                            win_h = min(chunk_size, height - y)
                            win_w = min(chunk_size, width - x)
                            win = Window(x, y, win_w, win_h)
                            
                            label_chunk_1d = np.full(win_h * win_w, background_out, dtype=np.uint8)

                            if use_mask:
                                # Aligned mask writes 1 for kept valid pixels
                                geo_mask = mask_src.read(1, window=win) == 1
                            else:
                                geo_mask = np.ones((win_h, win_w), dtype=bool)
                            
                            if not geo_mask.any():
                                dst.write(label_chunk_1d.reshape((win_h, win_w)), 1, window=win)
                                pbar.update(1)
                                if use_mask:
                                    del geo_mask
                                continue

                            chunk = src.read(window=win)
                            if isinstance(nodata_val, float) and np.isnan(nodata_val):
                                valid_mask = np.any(~np.isnan(chunk), axis=0)
                            else:
                                valid_mask = np.any(chunk != nodata_val, axis=0)

                            combined_mask_1d = (valid_mask & geo_mask).ravel()

                            if combined_mask_1d.any():
                                reshaped = chunk.reshape(bands, -1).T
                                valid_pixels = reshaped[combined_mask_1d].astype(np.float32)

                                preds = model.predict(valid_pixels)
                                mapped_preds = np.where(preds == target_class_in, target_class_out, background_out).astype(np.uint8) 
                                label_chunk_1d[combined_mask_1d] = mapped_preds
                                
                                del reshaped, valid_pixels, preds, mapped_preds

                            dst.write(label_chunk_1d.reshape((win_h, win_w)), 1, window=win)
                            pbar.update(1)
                            
                            del chunk, label_chunk_1d, combined_mask_1d
                            if use_mask:
                                del geo_mask
        finally:
            if mask_src is not None:
                mask_src.close()
                # Clean up the temporary aligned mask to save disk space
                Path(aligned_mask_path).unlink(missing_ok=True)

def execute_static_pipeline(
    base_dir: Path, 
    mask_path: Optional[Path], 
    input_shp_path: str,
    model_file: str, 
    delete_tiles: bool,
    use_mask: bool,
    mask_keep_values: Sequence[int],
    target_class_in: int,
    target_class_out: int,
    background_out: int,
    static_start: str,
    static_end: str,
    export_scale: int,
    tile_deg: float
) -> Path:
    
    base_dir.mkdir(parents=True, exist_ok=True)
    staging_dir = base_dir / "staging_temp"
    staging_dir.mkdir(parents=True, exist_ok=True)
    
    logger.info("Initiating STAC acquisition for optimal static composite into staging directory...")
    
    static_result = fetch_sentinel_static_imagery(
        aoi=input_shp_path,
        start=static_start,
        end=static_end,
        bands=["blue", "green", "red", "rededge1", "nir", "ndvi"],
        out_dir=str(staging_dir),
        res_m=export_scale,
        tile_deg=tile_deg,
        n_dates=2,
        dates=static_image_date,
        mask_clouds=False, 
        workers=4,
        build_vrt_mosaic=True,
        clip_to_aoi=False
    )
    
    raw_dates = static_result.get("dates", [])
    if not raw_dates:
        logger.warning("No 'dates' key found. Defaulting to 'Unknown_Date'.")
        date_suffix = "Unknown_Date"
    else:
        formatted_dates = [datetime.strptime(d, "%Y-%m-%d").strftime("%d_%b_%Y") for d in raw_dates]
        date_suffix = "_and_".join(formatted_dates)
        
    final_out_dir = base_dir / date_suffix
    final_output_path = final_out_dir / f"static_mosaic_{date_suffix}_Cls.tif"

    if final_output_path.exists():
        logger.info(f"[CHECKPOINT FOUND] Classification map for {date_suffix} already exists at: {final_output_path}")
        shutil.rmtree(staging_dir, ignore_errors=True)
        return final_output_path

    final_out_dir.mkdir(parents=True, exist_ok=True)
    
    logger.info(f"Renaming chunks and mapping to target directory: {final_out_dir.name}")
    new_chunk_paths = []
    
    for chunk in staging_dir.glob("*.tif"):
        new_chunk_name = f"{chunk.stem}_{date_suffix}{chunk.suffix}"
        new_chunk_path = final_out_dir / new_chunk_name
        shutil.move(str(chunk), str(new_chunk_path))
        new_chunk_paths.append(str(new_chunk_path))
        
    for meta_file in staging_dir.glob("*.json"):
        new_meta_name = f"{meta_file.stem}_{date_suffix}{meta_file.suffix}"
        shutil.move(str(meta_file), str(final_out_dir / new_meta_name))

    input_image_path = final_out_dir / f"static_mosaic_{date_suffix}.vrt"
    if new_chunk_paths:
        vrt_options = gdal.BuildVRTOptions(resampleAlg='nearest')
        gdal.BuildVRT(str(input_image_path), new_chunk_paths, options=vrt_options)

    shutil.rmtree(staging_dir, ignore_errors=True)
    
    # The mask is built exactly as before; only the classification step changes.
    aligned_mask_path = None
    if use_mask:
        aligned_mask_path = str(final_out_dir / f"{final_output_path.stem}_temp_mask.tif")
        create_aligned_mask(
            target_raster_path=str(input_image_path),
            raw_mask_path=str(mask_path),
            shp_path=input_shp_path,
            output_mask_path=aligned_mask_path,
            keep_values=mask_keep_values,
            chunk_size=2048,
        )

    # classify_raster reads its decision threshold from the model sidecar instead of
    # assuming 0.5, builds features from the same module training used, and scores the
    # image against the distribution the model was fitted on before classifying it.
    # It also writes a probability raster, so the threshold can be revisited later
    # without running inference over the AOI a second time.
    logger.info(f"Initiating classification pipeline for: {input_image_path}")
    probability_path = final_out_dir / f"static_mosaic_{date_suffix}_prob.tif"

    result = st_inference.classify_raster(
        raster_path=str(input_image_path),
        model_path=model_file,
        out_label_path=str(final_output_path),
        out_probability_path=str(probability_path),
        mask_path=aligned_mask_path,
        positive_out=target_class_out,
        background_out=background_out,
        enforce_domain=False,   # set True once the guard is trusted to stop a run
    )

    logger.info(f"Domain guard: {result.verdict}")
    logger.info(
        f"Threshold {result.threshold:.2f} taken from the sidecar; "
        f"{100 * result.positive_fraction:.1f}% of masked pixels classified as crop"
    )
    logger.info(f"Probability raster: {probability_path}")
    
    if final_output_path.exists():
        logger.info(f"Classification complete. GeoTIFF generated at: {final_output_path}")

    if delete_tiles:
        if final_output_path.exists():
            logger.info(f"--- CLEANING UP RAW STATIC STAC TILES AND VRT IN {final_out_dir.name} ---")
            for p in final_out_dir.rglob("*"):
                if p.is_file() and p.suffix in ['.tif', '.vrt', '.json']:
                    # Protect the final output from deletion
                    if p.absolute() != final_output_path.absolute():
                        try:
                            p.unlink(missing_ok=True)
                        except Exception as e:
                            logger.warning(f"Failed to delete {p.name}: {e}")
        else:
            logger.warning("--- SKIPPING RAW TILE CLEANUP: Final map was not produced. Tiles retained for debugging. ---")

    # --- FINAL SAFETY CHECK ---
    if not final_output_path.exists():
        logger.error("❌ PIPELINE FAILED: The final output map was NOT produced.")
        return None
                        
    return final_output_path

In [11]:
USE_MASK = True
DELETE_RAW_TILES = False

# We define the base directory; the final folder name is derived inside the pipeline
BASE_STATIC_DIR = Path(NEW_OUT_DIR)
MASK_RASTER_PATH = Path(sieved_map_path) if USE_MASK else None
MODEL_PATH = static_image_model

# Global placeholder for the final path
FINAL_STATIC_OUTPUT_PATH = None 

if __name__ == "__main__":
    if run_static_model:
        try:
            FINAL_STATIC_OUTPUT_PATH = execute_static_pipeline(
                base_dir=BASE_STATIC_DIR,
                mask_path=MASK_RASTER_PATH,
                input_shp_path=str(input_shp_path),
                model_file=MODEL_PATH,
                delete_tiles=DELETE_RAW_TILES,
                use_mask=USE_MASK,
                mask_keep_values=Target_Crop_Class,
                target_class_in=target_class_in,
                target_class_out=target_class_out,
                background_out=background_out,
                static_start=STATIC_START,
                static_end=STATIC_END,
                export_scale=EXPORT_SCALE,
                tile_deg=TILE_DEG
            )
        except Exception as e:
            logger.error(f"Pipeline failed during execution: {e}", exc_info=True)
        finally:
            release_os_memory()
            logger.info("System memory fully released.")
            logger.info(f"Final output accessible globally via FINAL_STATIC_OUTPUT_PATH: {FINAL_STATIC_OUTPUT_PATH}")
    else:
        logger.info("[Skipped] run_static_model is False. Bypassing Static Image Inference.")

2026-09-09 17:50:11,639 - __main__ - INFO - Initiating STAC acquisition for optimal static composite into staging directory...
2026-09-09 17:50:13,607 - farmdar.sentinel - INFO - AOI -> 1 tiles @ 10m (tile=0.10deg, bands=['blue', 'green', 'red', 'rededge1', 'nir', 'ndvi'], dates=['2026-08-30'], 8 workers)
2026-09-09 17:50:42,578 - farmdar.sentinel - INFO - [1/1] tile 0001: written (19.6s, 100.0% filled)
2026-09-09 17:50:42,767 - farmdar.sentinel - INFO - DONE 1 tiles in 0.5 min. dates=['2026-08-30'] out_dir=/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/staging_temp vrt=/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/staging_temp/static.vrt
2026-09-09 17:50:42,775 - __main__ - INFO - [CHECKPOINT FOUND] Classification map for 30_Aug_2026 already exists at: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/30_Aug_2026/static_mosaic_30_

# Noise Removal

In [22]:
import numpy as np
import rasterio
from rasterio.features import sieve
from scipy.ndimage import label, binary_dilation, generate_binary_structure
from pathlib import Path
from typing import List, Union

def apply_strict_directional_sieve(
    input_raster_path: Union[str, Path],
    target_classes: List[int],
    min_pixel_size: int = 15,
    connectivity: int = 4,
    nodata_val: int = 0
) -> str:
    """
    Applies a strict asymmetric Sieve filter using morphological connected components.
    A Non-Target clump is ONLY allowed to merge into the Target class group if it is 
    completely surrounded by Target pixels. Touching NoData or any other class aborts the merge.
    """
    in_path = Path(input_raster_path)
    out_name = f"{in_path.stem}_sieved_p{min_pixel_size}{in_path.suffix}"
    out_path = in_path.parent / out_name

    # ==========================================
    # SKIP EXISTING CHECK
    # ==========================================
    if out_path.exists():
        print(f"\n[Skipped] Sieved raster already exists at: {out_path}\n")
        return str(out_path)

    print(f"\nLoading categorical map for Strict Sieving: {in_path.name}")
    
    with rasterio.open(in_path) as src:
        meta = src.profile
        # Read as native dtype (int32 from our ML inference step)
        data = src.read(1)
        
    # Sieve requires the mask to be uint8 or boolean
    valid_mask = (data != nodata_val).astype(np.uint8)

    print(f"Phase 1: Base Sieve Filter (Removing blobs < {min_pixel_size} pixels)...")
    sieved_data = sieve(
        data, 
        size=min_pixel_size, 
        connectivity=connectivity, 
        mask=valid_mask
    )

    print("Phase 2: Enforcing Strict Topological Encapsulation...")
    
    # Define topological boundaries using the entire group of target classes
    is_target_orig = np.isin(data, target_classes)
    is_target_sieved = np.isin(sieved_data, target_classes)
    
    # 1. Identify all pixels that changed from Non-Target -> ANY Target Class
    changed_mask = (~is_target_orig) & is_target_sieved
    
    # 2. Define "Bad Neighbors": Any pixel in original data that is NOT in the target group.
    # We exclude the `changed_mask` pixels themselves so clumps don't flag themselves.
    bad_neighbors = (~is_target_orig) & ~changed_mask
    
    # 3. Create a structuring element that matches your sieve connectivity
    struct = generate_binary_structure(2, 1) if connectivity == 4 else generate_binary_structure(2, 2)
    
    # 4. Dilate the bad neighbors by 1 pixel to create a "collision zone"
    bad_borders = binary_dilation(bad_neighbors, structure=struct)
    
    # 5. Find pixels inside our changed clumps that intersect the collision zone
    touched_by_bad = changed_mask & bad_borders
    
    # 6. Assign a unique ID to every distinct clump in the changed_mask
    labeled_clumps, num_features = label(changed_mask, structure=struct)
    
    if num_features > 0:
        bad_labels = np.unique(labeled_clumps[touched_by_bad])
        bad_labels = bad_labels[bad_labels != 0] 
        
        if len(bad_labels) > 0:
            print(f"  -> Reverting {len(bad_labels)} illegal edge-merges...")
            revert_mask = np.isin(labeled_clumps, bad_labels)
            sieved_data[revert_mask] = data[revert_mask]

    # Strictly enforce NoData limits (do not let background merge into valid areas)
    sieved_data[data == nodata_val] = nodata_val

    print("Writing topologically-enforced output to disk...")
    meta.update(compress='lzw')
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(sieved_data, 1)

    print(f"SUCCESS: Strict Sieved raster saved to:\n{out_path}\n")
    return str(out_path)



In [23]:
if __name__ == "__main__":
    if run_static_model:
        try:
            # target_classes_to_preserve carries over from global params
            target_classes_to_preserve = [keep_label]
            
            sieved_tiff_path = apply_strict_directional_sieve(
                input_raster_path=FINAL_STATIC_OUTPUT_PATH,
                target_classes=target_classes_to_preserve,
                min_pixel_size=min_pixel_size,  
                connectivity=4,
                nodata_val=0
            )
        except NameError as e:
            print(f"ERROR: Required path variable is not defined: {e}")
        except Exception as e:
            print(f"ERROR: Second Sieving failed: {e}")
    else:
        print("\n[Skipped] run_static_model is False. Bypassing Second Sieve.\n")


Loading categorical map for Strict Sieving: static_mosaic_10_Aug_2026_Cls.tif
Phase 1: Base Sieve Filter (Removing blobs < 20 pixels)...
Phase 2: Enforcing Strict Topological Encapsulation...
Writing topologically-enforced output to disk...
SUCCESS: Strict Sieved raster saved to:
/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/10_Aug_2026/static_mosaic_10_Aug_2026_Cls_sieved_p20.tif



# Clipping and Relabeling

In [24]:
import os
import gc
import shutil
import tempfile
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
from pathlib import Path
from typing import Union, Optional, List

def vectorize_process_and_export(
    input_raster_path: Union[str, Path], 
    boundary_shp_path: Union[str, Path],
    output_dir: Union[str, Path],
    output_basename: str,
    target_labels: Union[int, List[int]], # <--- NOW ACCEPTS LIST OR INT
    relabel_as: int = 3015,
    min_area_acres: float = 0.5,
    save_shp_zip: bool = True
) -> Optional[str]:
    """
    End-to-end pipeline: Vectorize raster -> Clip to Boundary -> Dissolve -> 
    Explode (Multipart to Singlepart) -> Calculate Area -> Filter -> Export.
    """
    input_path = Path(input_raster_path)
    out_dir_path = Path(output_dir/"final_output")
    out_dir_path.mkdir(parents=True, exist_ok=True)
    
    out_base_path = out_dir_path / output_basename
    gpkg_output = f"{out_base_path}.gpkg"
    zip_output = f"{out_base_path}.zip"

    # ==========================================
    # SKIP EXISTING CHECK
    # ==========================================
    if Path(gpkg_output).exists() and (not save_shp_zip or Path(zip_output).exists()):
        print(f"\n[Skipped] Vectorized outputs already exist for: {output_basename}")
        return gpkg_output
    
    print(f"\n--- Starting Pipeline for: {output_basename} ---")
    
    # Standardize target_labels to a list
    if isinstance(target_labels, int):
        target_labels = [target_labels]
        
    print(f"1. Loading and vectorizing target classes {target_labels} from raster...")
    
    # ==========================================
    # 1. VECTORIZE RASTER
    # ==========================================
    with rasterio.open(input_path) as src:
        image = src.read(1)
        transform = src.transform
        raster_crs = src.crs

    # HIGHEST PERFORMANCE MASKING: np.isin creates a boolean mask for all valid 
    # classes instantly. This merges all maize classes into one monolithic binary 
    # map in RAM before vectorization, saving massive compute time on dissolve.
    target_mask = np.isin(image, target_labels)
    
    if not target_mask.any():
        print(f"  [Warning] No pixels found for classes {target_labels}. Aborting.")
        return None

    # Generate geometries
    geom_generator = shapes(image, mask=target_mask, transform=transform)
    features = [{'geometry': shape(geom_dict)} for geom_dict, _ in geom_generator]
    
    gdf = gpd.GeoDataFrame(features, crs=raster_crs)
    print(f"  -> Extracted {len(gdf)} raw polygon features.")

    # Free up memory
    del image, target_mask
    gc.collect()

    # ==========================================
    # 2. LOAD & PREP BOUNDARY
    # ==========================================
    print(f"2. Loading boundaries for clipping...")
    boundary_gdf = gpd.read_file(boundary_shp_path)
    
    if boundary_gdf.empty:
        print(f"  [Error] The provided boundary shapefile is empty. Aborting.")
        return None
        
    boundary_proj = boundary_gdf.to_crs(raster_crs)

    # ==========================================
    # 3. GEOPROCESSING (Clean, Clip, Dissolve, Explode)
    # ==========================================
    print("3. Executing GEOS Geoprocessing (Clean, Clip, Dissolve, Singlepart)...")
    
    gdf['geometry'] = gdf.geometry.make_valid()
    gdf = gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])]
    
    clipped_gdf = gpd.clip(gdf, boundary_proj)
    if clipped_gdf.empty:
        print(f"  [Warning] No features intersect with the boundary. Aborting.")
        return None
        
    dissolved_gdf = clipped_gdf.dissolve()
    
    singlepart_gdf = dissolved_gdf.explode(index_parts=False).reset_index(drop=True)
    singlepart_gdf = singlepart_gdf[singlepart_gdf.geom_type == 'Polygon']
    singlepart_gdf = singlepart_gdf[~singlepart_gdf.is_empty]

    # ==========================================
    # 4. AREA CALCULATION & FILTERING
    # ==========================================
    print(f"4. Calculating Area (UTM 42N) and filtering (>= {min_area_acres} acres)...")
    
    singlepart_utm = singlepart_gdf.to_crs(epsg=32642)
    singlepart_gdf['area_acres'] = singlepart_utm.geometry.area * 0.000247105
    
    singlepart_gdf = singlepart_gdf[singlepart_gdf['area_acres'] >= min_area_acres]
    
    if singlepart_gdf.empty:
        print("  [Warning] No features remained after area filtering. Aborting.")
        return None
        
    # ==========================================
    # 5. ASSIGN FINAL LABEL & PRINT METRICS
    # ==========================================
    singlepart_gdf['predicted'] = relabel_as
    singlepart_gdf = singlepart_gdf[['predicted', 'area_acres', 'geometry']]
    
    total_acres = singlepart_gdf['area_acres'].sum()
    print(f"\n✅ Processing Complete!")
    print(f"   Retained Features: {len(singlepart_gdf):,}")
    print(f"   Total Final Area:  {total_acres:,.2f} acres")
    print(f"   Final Class Label: {relabel_as}")

    # ==========================================
    # 6. EXPORT
    # ==========================================
    print(f"\n5. Exporting outputs...")
    singlepart_gdf.to_file(gpkg_output, driver="GPKG")
    print(f"   -> Saved GPKG: {gpkg_output}")
    
    if save_shp_zip:
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_shp_path = Path(tmpdir) / f"{output_basename}.shp"
            singlepart_gdf.to_file(tmp_shp_path, driver="ESRI Shapefile")
            
            shutil.make_archive(
                base_name=str(out_base_path), 
                format='zip', 
                root_dir=tmpdir
            )
        print(f"   -> Saved ZIP (Shapefile): {zip_output}")

    del gdf, clipped_gdf, dissolved_gdf, singlepart_gdf, singlepart_utm
    gc.collect()
    
    return gpkg_output




In [25]:
if __name__ == "__main__":
    try:
        from pathlib import Path
        
        CUSTOM_BASENAME = final_output_name
        
        # ---------------------------------------------------------
        # ROUTER LOGIC: Decide inputs based on master toggle
        # ---------------------------------------------------------
        if run_static_model:
            print("\n[Routing] Static Model Pipeline Enabled. Using Static Inference outputs.")
            target_raster = sieved_tiff_path       # From 2nd Sieve (Static model output)
            labels_to_extract = keep_label         # Just 3
        else:
            print("\n[Routing] Static Model Pipeline Bypassed. Using Direct NDVI outputs.")
            target_raster = sieved_map_path        # From 1st Sieve (NDVI model output)
            labels_to_extract = Target_Crop_Class      # [1, 4, 5, 6, 7]
            
        OUTPUT_DIR = Path(target_raster).parent
        
        final_vector_path = vectorize_process_and_export(
            input_raster_path=target_raster,    
            boundary_shp_path=input_shp_path,      
            output_dir=OUTPUT_DIR,
            output_basename=CUSTOM_BASENAME,
            target_labels=labels_to_extract,       # Dynamically passes int or list
            relabel_as=relabel_as,                 # Always 3015
            min_area_acres=min_area_acres,
            save_shp_zip=save_shp_zip
        )
        
    except NameError as e:
        print(f"ERROR: A required variable is missing in the environment. {e}")


[Routing] Static Model Pipeline Enabled. Using Static Inference outputs.

--- Starting Pipeline for: almoiz_unit_1_test_feature_1_cane_2026 ---
1. Loading and vectorizing target classes [1] from raster...
  -> Extracted 801 raw polygon features.
2. Loading boundaries for clipping...
3. Executing GEOS Geoprocessing (Clean, Clip, Dissolve, Singlepart)...


2026-09-09 13:09:31,167 - pyogrio._io - INFO - Created 744 records


4. Calculating Area (UTM 42N) and filtering (>= 0.5 acres)...

✅ Processing Complete!
   Retained Features: 744
   Total Final Area:  5,923.40 acres
   Final Class Label: 1

5. Exporting outputs...
   -> Saved GPKG: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/10_Aug_2026/final_output/almoiz_unit_1_test_feature_1_cane_2026.gpkg


2026-09-09 13:09:31,294 - pyogrio._io - INFO - Created 744 records


   -> Saved ZIP (Shapefile): /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/10_Aug_2026/final_output/almoiz_unit_1_test_feature_1_cane_2026.zip
